In [7]:
import cv2
import numpy as np
import os
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Hand detector
hand_base = python.BaseOptions(model_asset_path="hand_landmarker.task")
hand_options = vision.HandLandmarkerOptions(base_options=hand_base, num_hands=2)
hand_detector = vision.HandLandmarker.create_from_options(hand_options)

# Pose detector
pose_base = python.BaseOptions(model_asset_path="pose_landmarker.task")
pose_options = vision.PoseLandmarkerOptions(base_options=pose_base)
pose_detector = vision.PoseLandmarker.create_from_options(pose_options)

In [8]:
def mediapipe_detection(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    hand_results = hand_detector.detect(mp_image)
    pose_results = pose_detector.detect(mp_image)

    return hand_results, pose_results

In [9]:
def extract_pose(pose_results):
    if pose_results.pose_landmarks:
        pose = np.array(
            [[lm.x, lm.y, lm.z, lm.visibility]
             for lm in pose_results.pose_landmarks[0]]
        ).flatten()
    else:
        pose = np.zeros(33*4)
    return pose


def extract_hands(hand_results):
    left = np.zeros(21*3)
    right = np.zeros(21*3)

    if hand_results.hand_landmarks:
        for idx, hand in enumerate(hand_results.hand_landmarks):
            hand_array = np.array([[lm.x, lm.y, lm.z] for lm in hand]).flatten()
            if idx == 0:
                left = hand_array
            elif idx == 1:
                right = hand_array

    return left, right


def extract_keypoints(hand_results, pose_results):
    pose = extract_pose(pose_results)
    left, right = extract_hands(hand_results)
    return np.concatenate([pose, left, right])

In [10]:
VIDEO_PATH = "selected_videos"
DATA_PATH = "MP_Data"

actions = [f"sign_{i}" for i in range(60)]
no_sequences = 30
sequence_length = 30

for action in actions:
    for sequence in range(no_sequences):
        os.makedirs(os.path.join(DATA_PATH, action, str(sequence)), exist_ok=True)

print("Folder Structure Created")

Folder Structure Created


In [12]:
video_files = [f for f in os.listdir(VIDEO_PATH) if f.endswith(".mp4")]

print("Videos found:", len(video_files))

for idx, video_file in enumerate(video_files):

    action = actions[idx]
    video_path = os.path.join(VIDEO_PATH, video_file)

    cap = cv2.VideoCapture(video_path)

    for sequence in range(no_sequences):
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)  # restart video

        for frame_num in range(sequence_length):

            ret, frame = cap.read()
            if not ret:
                break
            hand_results, pose_results = mediapipe_detection(frame)

            keypoints = extract_keypoints(hand_results, pose_results)

            npy_path = os.path.join(DATA_PATH, action, str(sequence), str(frame_num))
            np.save(npy_path, keypoints)

    cap.release()

print("TEST DONE")

Videos found: 60
TEST DONE
